# Diagnosing diabetes with KNN

In this small notebook we use data about diabetes in Pima Indian women to see how well we can predict diabetes given variables like age, glucose level, blood pressure, and skin thickness.

We focus on two things:
- data exploration and cleaning
- using hyperparameter tuning to find the best hyperparameter values for KNN classification

I downloaded the dataset from Kaggle on October 21, 2021.

https://www.kaggle.com/uciml/pima-indians-diabetes-database

Information about the data set can be found on the Kaggle page.

v1.7

### Instructions

- Please read the entire notebook carefully!
- Note that plots are preceded by a question and followed by interpretation of the plot.
- Each problem cell begins with #@.
- Do not make changes outside the problem cells.
- Be sure to include plot titles, labels, etc. as shown in model output.
- Use plotting methods covered in class.
- Run your code from top to bottom before submitting, otherwise points will be deducted.
- Do not modify the file name.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# plotting
sns.set_theme(context='notebook', style='whitegrid')
plt.rcParams['figure.figsize'] = (4,3)   # default plot size

In [ ]:
# suppress "future warnings"
warnings.simplefilter(action='ignore', category=FutureWarning)

### Set the random seed for repeatability

In [ ]:
np.random.seed(0)

### Read the data

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/grbruns/cst383/master/diabetes.csv")

It is useful to identify the predictor and target variables right away.

In [ ]:
target = 'Outcome'
predictors = list(df.columns)
predictors.remove(target)

### Data exploration

Looking at an overview of the data, we see no NA values.  All variables are numeric.

In [ ]:
df.info()

It is helpful to look at a little of the raw data.

In [ ]:
df.head()

'Outcome' is the target value.  A value of 1 indicates the presence of diabetes.  

How many people represented in the data have diabetes?

In [ ]:
#@ 1  Create a bar plot that shows the fraction of patients with diabetes, and without diabetes.
# For this and all later problems, use visualization methods covered in class.

# YOUR CODE HERE

About 1/3 of the patients have diabetes, according to the data.

What are the basic statistics of the numeric variables?

In [ ]:
df.describe().round(2)

We see that there are no negative values in the dataset.  

Something that looks odd is that many variables have zero as their minimum value.  Can blood pressure really be zero?  What about skin thickness, and body mass index?  Perhaps some of the zeroes indicate bad data.

Does the data contain outliers?  This will be easier to see if the data is scaled.

In [ ]:
#@ 2 Produce describe() output after using zscore normalization on each column.
# Do not modify df, as it will be used in later cells.
# Hint: in class we covered how to zscore normalize all columns of a dataframe.

# YOUR CODE HERE

We see that the maximum insulin value is more than 6 standard deviations above the mean, and the max skin thickness and diabetes pedigree function values are also large.

Box plots can help identify outliers.  When showing multiple boxplots at once, scaling is useful.

In [ ]:
#@ 3 Produce box plots for all predictor values.  Scale each predictor using Z-score normalization.
# Hint: recall from class how a boxplot can be created for a dataframe.

# YOUR CODE HERE

There seem to be many outliers in the Insulin and DiabetesPedegreeFunction variables.  

We can get a deeper feeling for zero values and outliers by plotting the data.

In [ ]:
#@ 4 Produce a plot containing scatterplots for all pairs of predictor variables.
# Hint: it takes a while for the output to be produced, so don't worry about that.

# YOUR CODE HERE

We find that some of the 0 values do indeed look strange.  For example, the zero BMI values look strange.  Also, some of the max values look like outliers.  For example, the largest skin thickness value.

### Investigating zero values

A concern is that zero values might represent missing data.

Let's focus first on insulin.  What is the distribution of insulin values?

In [ ]:
plt.hist(df['Insulin']);
plt.title("Histogram of insulin values");

The distribution is highly skewed.  Plotting the log may make the distribution clearer.

In [ ]:
plt.hist(np.log10(df['Insulin']+1));
plt.title("Histogram of insultin values (log10 scale)");

This picture makes the zero insulin values look very suspicious.

What about the other predictors?  For each predictor, what fraction of the values are 0?

In [ ]:
#@ 5  Write an expression to compute the fraction of values of each predictor that are zero.
# Put output in decreasing order of 0 fraction.
# Hint: don't forget the 'predictors' variable defined above.
# Hint: you can use round().

# YOUR CODE HERE

About 46% of the insulin values are zero.  This could suggest they are valid.  Also, a little background research suggests zero insulin values are possible.

Another way to understand zero values is to see how many standard deviations they are away from the mean value.

In [ ]:
df[predictors].apply(lambda x: x.mean()/x.std()).sort_values(ascending=False).round(3)

This backs up the idea that blood pressure values of 0 represent missing data.  

Are the 0 values in the data set clustered in some rows?
In other words, are the 0's spread evenly across people, or clustered in some people?  To look into this, we can count the number of rows with no zero values, with 1 zero value, etc.

In [ ]:
#@ 6  Compute the number of *rows* that contain *no* predictor values of 0,
# the number of rows that contain *one* predictor value of 0, etc.
# Show the results as a bar plot.
# Hint: it's possible to take the sum of every *row* of a boolean dataframe.
# We did in class when counting NAs by row.

# YOUR CODE HERE

We see that about 170 rows contain two zero values, but very few rows contain more than two zero values.

In the rows with more than one zero value, which predictors are 0?

In [ ]:
df[(df[predictors] == 0).sum(axis=1) > 1].head(10)

Zero values for skin thickness and insulin seem to go together.  Does a scatter plot confirm this idea?

In [ ]:
sns.scatterplot(data=df, x='SkinThickness', y='Insulin')
plt.xlabel('skin thickness')
plt.title('Skin thickness by insulin value');

It seems that if skin thickness is 0, then insulin is 0, because there are no skin thickness values of 0 except where insulin is 0.

Perhaps if a person's skin is very thin, it is hard to test for insulin.  Talking to a diabetes specialist would help in understanding this.

### Data preprocessing

Our strategy on zero values will be to remove rows in which BMI, Glucose, BloodPressure, or SkinThickness are 0.  An alternative approach would be to impute values for these zero values.

In [ ]:
#@ 7 Modify df to to remove the rows in which BMI, Glucose, BloodPressure, or SkinThickness have value 0.
# Hint: use a boolean mask that involves these variables.
# Hint: write an assignment statement.
# Don't forget to stick to techniques covered in class.

# YOUR CODE HERE

Use describe again to see the result of removing these rows.

In [ ]:
df.describe()

Put the predictor and target values into NumPy arrays.

In [ ]:
X = df[predictors].values
y = df[target].values

In [ ]:
print('X shape: {}'.format(X.shape))
print('y shape: {}'.format(y.shape))

Perform an 75/25 train/test split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

Scale the data using z-score normalization.  Note that the scaler is trained on the training data, and the trained scaler is used on both the training and test data.  The target values are not scaled.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Basic KNN classification

In [ ]:
#@ 8  Train a KNN classifier on the training data.
# Create a KNeighborsClassifier object and store it as variable 'knn'.
# Use the default hyperparameters (in other words, don't specify any hyperparamters).
# Then train the classifier on the *training* data.
#
# Hint: I expect two lines of code here.  Use a semicolon at the
# end of the line on which you train the classifier to suppress
# the unneeded output.

# YOUR CODE HERE

Three important hyperparameters of a KNN classifier:
- the number of nearest neighbors ('n_neighbors' in KNeighborsClassifier)
- the distance metric  ('metric' and 'p' in KNeighborsClassifier)
- whether closer neighbors should be more important when making predictions ('weights' in KNeighborsClassifier)

If 'metric' is 'minkowski', then p = 1 means manhattan distance, and p = 2 means Euclidian distance.

Let's look at the default hyperparameters for the KNN classifier.

In [ ]:
# examining default parameters
knn.get_params()

What cross-validation accuracy is achieved with the default KNN classifier?

In [ ]:
#@ 9  Compute the cross-validation accuracy of the classifier.
# Use 10 folds in cross validation.
# Use the method covered in class, of course.
# Store your result as variable 'cv_accuracy'.

# YOUR CODE HERE

In [ ]:
print('CV accuracy using default hyperparameters: {:.3f}'.format(cv_accuracy))

### Compute the accuracy we'd get by always predicting "no diabetes".

When we compute the accuracy of a classifier, we want a baseline for comparison.  The usual baseline is the accuracy that you would get if you always predicted the most common value of the predictor variable.  In this case, the most common value of y_test is 0.

In [ ]:
#@ 10  Compute the baseline accuracy by computing the fraction of the y_train values that are 0.
# Store the result as variable 'baseline_accuracy'.
# Hint: here it is okay to assume the majority target value is 0, but in future we will always calculate it.

# YOUR CODE HERE

In [ ]:
print('Baseline accuracy: {:.3f}'.format(baseline_accuracy))

So the test accuracy is significantly better than the baseline.

### Determine best k by using 10-fold cross validation

The default value of k (called n_neighbors in Scikit-Learn) is 5 with KNeighborsClassifier.  Is this a good value for k?

In [ ]:
#@ 11  Compute 10-fold cross-validation accuracy for k=1, 3, 5, ..., 23.
# Store the accuracy values in list cv_accuracy.
# Store the k values in list ks.
# Hint: we covered this in class.  The cv-accuracy value should be the mean
# of the accuracy values for the 10 folds.

# YOUR CODE HERE

In [ ]:
plt.plot(ks, cv_accuracy)
plt.title('KNN accuracy by k (diabetes data)')
plt.xlabel('k')
plt.ylabel('cross-validation accuracy');

The plot shows that a value of 11 is best, but 19 is almost as good.  Perhaps any value between about 11 and 21 is good.

### Find best combination of hyperparameters k and p using grid search

The problem with finding the best k value on its own, is that the best value of k might depend on the other hyperparameter values, such as distance function.

We really would like to search for the best combination of parameter values.  Grid search is a good way to do this.

In [ ]:
#@ 12  Use GridSearchCV to find the best settings for hyperparameters 'n_neighbors',
# 'weights', and 'p' of KNeighborsClassifier.
# For 'n_neighbors', consider values 5, 7, 9, ..., 15.
# For 'weights', consider values 'uniform' and 'distance'.
# For 'p', consider values 1 and 2.
# Assign your GridSearchCV object to variable 'knn_cv', and fit it using the training data.
# You do not need to do more than to do the fit here.

# YOUR CODE HERE

In [ ]:
print(knn_cv.best_params_)

In [ ]:
print('best CV accuracy: {:.3f}'.format(knn_cv.best_score_))

The CV accuracy we get with the best hyperparameter values is better than with the default hyperparameters.

### Train model with best hyperparameters on all training data.

After using cross validation, we want to train the best model (in other words, the model with the best hyperparameter values) on the complete set of training data.

Fortunately, GridSearchCV() does this for us automatically in the .fit() method, as long as the 'refit' parameter of GridSearchCV is True, which it is by default.

### Compute test accuracy using the score() function

We have not used the test data yet.  We now compute test accuracy to see how our classifier does on data never seen before.

In [ ]:
#@ 13  Compute the test accuracy of the classifier with the best
# hyperparameter values using the test data.
# Use the score() function to do this, and store the
# result in variable 'test_accuracy'.

# YOUR CODE HERE

In [ ]:
print('Test accuracy: {:.3f}'.format(test_accuracy))

Our test accuracy is significantly better than our baseline accuracy, but only by about 10%.  

Some interesting questions:
- Could we do better with a different type of classification algorithm?
- Would we have done better if we had imputed the zero values?
- Which of the predictor values are most important?
- Could we do better if we drop some of the predictor variables?